# AURA-JKN: Synthetic Data Generation & Heterogeneous Graph Construction

This notebook generates a synthetic BPJS Kesehatan claims dataset (Healthkathon 2026) and constructs a heterogeneous graph representation using PyTorch Geometric (`HeteroData`).

### Pipeline Overview:
1. **Facility (Faskes) Generation**: FKTP & FKRTL healthcare facilities across Indonesia.
2. **Doctor Generation**: Medical doctors with specialties, home facilities, and schedule allocations.
3. **Patient Generation**: Pseudonymized identity hashes using salted SHA-256 for privacy compliance.
4. **Claim Simulation**: 80% legitimate claims and 20% synthetic fraud scenarios:
   - **Scenario A**: Fake Patient Cluster (phantom claims concentrated on a single doctor/facility without genuine referrals).
   - **Scenario B**: Extreme Doctor Volume (statistically anomalous daily claim counts far exceeding peer capacity).
   - **Scenario C**: Spatio-Temporal Conflict (impossible concurrent/near-concurrent service across geographically distant facilities).
5. **Parquet Export**: Columnar entity storage for analytical pipelines.
6. **Heterogeneous Graph Construction**: `HeteroData` nodes (`patient`, `doctor`, `facility`, `claim`, `diagnosis`) and relational edges.
7. **Validation & Serialization**: Verification of tensor dimensions and saving to `aura_jkn_hetero_graph.pt`.

### Dependency Installation
If running in a fresh environment, ensure the required dependencies are installed:
```bash
pip install torch torch-geometric pandas numpy pyarrow
```

In [22]:
%%bash
pip install -q torch-geometric

In [23]:
import hashlib
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
from torch_geometric.data import HeteroData

# Set deterministic seeds for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print(f"Libraries loaded successfully. Seed set to {RANDOM_SEED}.")

Libraries loaded successfully. Seed set to 42.


## 1. Healthcare Facilities (Faskes)

In [24]:
n_facilities = 30
facility_ids = [f"FAC{i:04d}" for i in range(n_facilities)]
facility_classes = np.random.choice(["FKTP", "FKRTL"], size=n_facilities, p=[0.7, 0.3])

# Coordinates bounded roughly within Indonesian territory
latitudes = np.random.uniform(-6.0, 6.0, size=n_facilities)
longitudes = np.random.uniform(95.0, 141.0, size=n_facilities)

facilities_df = pd.DataFrame({
    "facility_id": facility_ids,
    "facility_class": facility_classes,
    "latitude": latitudes,
    "longitude": longitudes
})

facilities_df.head()

,facility_id,facility_class,latitude,longitude
0,FAC0000,FKTP,1.290538,112.879155
1,FAC0001,FKRTL,-3.953711,107.482055
2,FAC0002,FKRTL,-5.219381,133.121925
3,FAC0003,FKTP,5.386626,111.410653
4,FAC0004,FKTP,5.587584,107.922987


## 2. Doctor Directory

In [25]:
n_doctors = 150
doctor_ids = [f"DOC{i:04d}" for i in range(n_doctors)]
sip_numbers = [f"SIP{np.random.randint(1000000, 9999999)}" for _ in range(n_doctors)]
specialties = np.random.choice(["Umum", "Pediatri", "Kardio", "Ortho", "Kulit"], size=n_doctors)
home_facility_ids = np.random.choice(facility_ids, size=n_doctors)
schedule_hours = np.random.randint(20, 51, size=n_doctors)

doctors_df = pd.DataFrame({
    "doctor_id": doctor_ids,
    "sip_number": sip_numbers,
    "specialty": specialties,
    "home_facility_id": home_facility_ids,
    "schedule_hours": schedule_hours
})

doctors_df.head()

,doctor_id,sip_number,specialty,home_facility_id,schedule_hours
0,DOC0000,SIP6401218,Ortho,FAC0003,42
1,DOC0001,SIP7914550,Kardio,FAC0001,48
2,DOC0002,SIP7167294,Umum,FAC0019,30
3,DOC0003,SIP7906726,Umum,FAC0009,30
4,DOC0004,SIP6888794,Ortho,FAC0025,37


## 3. Patient Master Data (Pseudonymized)

Patient identity is protected via a salted SHA-256 hash across synthetic NIK and name pairs.

In [26]:
n_patients = 5000
nik_base = np.random.randint(10**15, 10**16 - 1, size=n_patients).astype(str)
name_prefixes = ["Budi", "Ani", "Citra", "Dedi", "Eka", "Fani", "Gita", "Hadi"]
name_suffixes = ["Santoso", "Wijaya", "Putri", "Nur", "Setiawan", "Lestari", "Kusuma", "Dewi"]
names = [f"{np.random.choice(name_prefixes)} {np.random.choice(name_suffixes)}" for _ in range(n_patients)]

SALT = b"AURA-JKN-SALT-2026"

def hash_patient(nik: str, name: str) -> str:
    payload = f"{nik}|{name}".encode("utf-8")
    return hashlib.sha256(SALT + payload).hexdigest()

patient_ids = [hash_patient(nik, name) for nik, name in zip(nik_base, names)]
home_regions = np.random.choice([f"PROV{i:02d}" for i in range(1, 35)], size=n_patients)
referral_history_ids = ["" for _ in range(n_patients)]

patients_df = pd.DataFrame({
    "patient_id": patient_ids,
    "nik": nik_base,
    "name": names,
    "home_region": home_regions,
    "referral_history_id": referral_history_ids
})

patients_df.head()

,patient_id,nik,name,home_region,referral_history_id
0,3b28578ba0644416fc6573ada79167def0259202810cb5...,5479189974158043,Ani Wijaya,PROV16,
1,90ca5360ba5e1090024fdf36b39748f77d9887b691f1d5...,9297941938815900,Hadi Santoso,PROV25,
2,577006d852812f4d171cae3100fd60e5e9da0d3c385931...,7873652393051693,Dedi Kusuma,PROV26,
3,e19ea1f8d610558748a025b5503b01f67d16a462a04498...,1525716119735548,Gita Santoso,PROV09,
4,e54267b723e120126a3439072e36809a3e07ca952534e0...,2884038728390225,Eka Putri,PROV15,


## 4. Claims Simulation (Normal & Fraud Scenarios)

- 80% Normal baseline claims
- 20% Fraudulent claims split across Scenarios A, B, and C

In [27]:
n_claims_total = 20000
n_normal = int(0.8 * n_claims_total)
n_fraud = n_claims_total - n_normal
n_fraud_each = n_fraud // 3

icd10_list = ["A00", "B20", "I10", "E11", "J45", "M54", "L20", "N18", "K29", "R07"]
icd9_list = ["001", "042", "401", "250", "493", "724", "692", "585", "530", "786"]

def random_service_datetime() -> pd.Timestamp:
    day_offset = np.random.randint(0, 365)
    base_date = pd.Timestamp("2024-01-01") + pd.Timedelta(days=day_offset)
    return base_date.replace(
        hour=int(np.random.randint(0, 24)),
        minute=int(np.random.randint(0, 60)),
        second=int(np.random.randint(0, 60))
    )

print("Generating legitimate claims...")
service_dt_norm = [random_service_datetime() for _ in range(n_normal)]
claims_norm = pd.DataFrame({
    "claim_id": [f"CLM{i:08d}" for i in range(n_normal)],
    "patient_id": np.random.choice(patients_df["patient_id"], size=n_normal),
    "doctor_id": np.random.choice(doctors_df["doctor_id"], size=n_normal),
    "facility_id": np.random.choice(facility_ids, size=n_normal),
    "icd10_code": np.random.choice(icd10_list, size=n_normal),
    "icd9_code": np.random.choice(icd9_list, size=n_normal),
    "service_date": [dt.date() for dt in service_dt_norm],
    "service_time": [dt.time() for dt in service_dt_norm],
    "ina_cbgs_tariff": np.random.uniform(50000, 1500000, size=n_normal).round(0),
    "length_of_stay": np.random.randint(0, 15, size=n_normal),
    "is_fraud": np.zeros(n_normal, dtype=int),
    "fraud_scenario": [None] * n_normal
})

print("Generating Fraud Scenario A (Fake Patient Cluster)...")
cluster_region = np.random.choice(patients_df["home_region"].unique())
cluster_patients = patients_df[patients_df["home_region"] == cluster_region].sample(n=50, random_state=RANDOM_SEED)
cluster_doctor = np.random.choice(doctors_df["doctor_id"])
cluster_facility = np.random.choice(facility_ids)
service_dt_A = [random_service_datetime() for _ in range(n_fraud_each)]

claims_A = pd.DataFrame({
    "claim_id": [f"CLMA{i:08d}" for i in range(n_fraud_each)],
    "patient_id": np.random.choice(cluster_patients["patient_id"], size=n_fraud_each),
    "doctor_id": np.full(n_fraud_each, cluster_doctor),
    "facility_id": np.full(n_fraud_each, cluster_facility),
    "icd10_code": np.random.choice(icd10_list, size=n_fraud_each),
    "icd9_code": np.random.choice(icd9_list, size=n_fraud_each),
    "service_date": [dt.date() for dt in service_dt_A],
    "service_time": [dt.time() for dt in service_dt_A],
    "ina_cbgs_tariff": np.random.uniform(50000, 1500000, size=n_fraud_each).round(0),
    "length_of_stay": np.random.randint(0, 15, size=n_fraud_each),
    "is_fraud": np.ones(n_fraud_each, dtype=int),
    "fraud_scenario": ["A"] * n_fraud_each
})

print("Generating Fraud Scenario B (Extreme Doctor Volume)...")
fraud_doc_B = np.random.choice(doctors_df["doctor_id"])
service_dt_B = [random_service_datetime() for _ in range(n_fraud_each)]

claims_B = pd.DataFrame({
    "claim_id": [f"CLMB{i:08d}" for i in range(n_fraud_each)],
    "patient_id": np.random.choice(patients_df["patient_id"], size=n_fraud_each),
    "doctor_id": np.full(n_fraud_each, fraud_doc_B),
    "facility_id": np.random.choice(facility_ids, size=n_fraud_each),
    "icd10_code": np.random.choice(icd10_list, size=n_fraud_each),
    "icd9_code": np.random.choice(icd9_list, size=n_fraud_each),
    "service_date": [dt.date() for dt in service_dt_B],
    "service_time": [dt.time() for dt in service_dt_B],
    "ina_cbgs_tariff": np.random.uniform(50000, 1500000, size=n_fraud_each).round(0),
    "length_of_stay": np.random.randint(0, 15, size=n_fraud_each),
    "is_fraud": np.ones(n_fraud_each, dtype=int),
    "fraud_scenario": ["B"] * n_fraud_each
})

print("Generating Fraud Scenario C (Spatio-Temporal Conflict)...")
fraud_doc_C = np.random.choice(doctors_df["doctor_id"])
facility_coords = facilities_df.set_index("facility_id")[["latitude", "longitude"]]
facility_id_C1 = np.random.choice(facility_ids)
candidates_C2 = [fid for fid in facility_ids if fid != facility_id_C1]
np.random.shuffle(candidates_C2)

# Find a facility with significant geographical distance (> 5 degrees)
facility_id_C2 = candidates_C2[0]
for cand in candidates_C2:
    lat_diff = abs(facility_coords.loc[facility_id_C1, "latitude"] - facility_coords.loc[cand, "latitude"])
    lon_diff = abs(facility_coords.loc[facility_id_C1, "longitude"] - facility_coords.loc[cand, "longitude"])
    if lat_diff >= 5.0 or lon_diff >= 5.0:
        facility_id_C2 = cand
        break

facility_id_C = np.where(np.arange(n_fraud_each) % 2 == 0, facility_id_C1, facility_id_C2)
base_dt_C = [random_service_datetime() for _ in range(n_fraud_each // 2)]
service_dt_C = [dt for pair_dt in base_dt_C for dt in (pair_dt, pair_dt)]
if len(service_dt_C) < n_fraud_each:
    service_dt_C.append(random_service_datetime())

claims_C = pd.DataFrame({
    "claim_id": [f"CLMC{i:08d}" for i in range(n_fraud_each)],
    "patient_id": np.random.choice(patients_df["patient_id"], size=n_fraud_each),
    "doctor_id": np.full(n_fraud_each, fraud_doc_C),
    "facility_id": facility_id_C,
    "icd10_code": np.random.choice(icd10_list, size=n_fraud_each),
    "icd9_code": np.random.choice(icd9_list, size=n_fraud_each),
    "service_date": [dt.date() for dt in service_dt_C],
    "service_time": [dt.time() for dt in service_dt_C],
    "ina_cbgs_tariff": np.random.uniform(50000, 1500000, size=n_fraud_each).round(0),
    "length_of_stay": np.random.randint(0, 15, size=n_fraud_each),
    "is_fraud": np.ones(n_fraud_each, dtype=int),
    "fraud_scenario": ["C"] * n_fraud_each
})

# Merge and shuffle
claims_df = pd.concat([claims_norm, claims_A, claims_B, claims_C], ignore_index=True)
claims_df = claims_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print(f"Total claims: {len(claims_df):,}")
print(f"Total fraudulent claims: {claims_df['is_fraud'].sum():,}")
print("Fraud breakdown:")
print(claims_df["fraud_scenario"].value_counts(dropna=False))
claims_df.head()

Generating legitimate claims...
Generating Fraud Scenario A (Fake Patient Cluster)...
Generating Fraud Scenario B (Extreme Doctor Volume)...
Generating Fraud Scenario C (Spatio-Temporal Conflict)...
Total claims: 19,999
Total fraudulent claims: 3,999
Fraud breakdown:
fraud_scenario
None    16000
C        1333
A        1333
B        1333
Name: count, dtype: int64


,claim_id,patient_id,doctor_id,facility_id,icd10_code,icd9_code,service_date,service_time,ina_cbgs_tariff,length_of_stay,is_fraud,fraud_scenario
0,CLM00010650,2b0e84013babe73a9ff3349cd26b015c81d91dbb2965a0...,DOC0086,FAC0000,R07,724,2024-09-12,03:53:08,461876.0,9,0,None
1,CLM00002041,a46ba9c77fc474cf0d43dc52164fea9502861bac9fad8c...,DOC0078,FAC0017,E11,401,2024-11-06,07:59:56,613541.0,10,0,None
2,CLM00008668,b6afbb2d7a316e156d320e66486699c93353001c9a7762...,DOC0103,FAC0026,M54,585,2024-05-08,06:57:33,253452.0,11,0,None
3,CLM00001114,a4539304ef5ad06ec9887d3f7f9d2f1770be557c9107c6...,DOC0052,FAC0018,M54,493,2024-06-10,20:34:41,565371.0,11,0,None
4,CLM00013902,e92c38c1a59218a3a06633fb36a5747e4bcbe83e6e2eff...,DOC0024,FAC0017,B20,250,2024-07-20,06:08:54,1042337.0,8,0,None


## 5. Export to Parquet

Writing structured dataframes into Parquet format for fast loading and schema enforcement.

In [28]:
# Cast date and time to strings or timestamps for clean Parquet export compatibility
claims_export_df = claims_df.copy()
claims_export_df["service_date"] = claims_export_df["service_date"].astype(str)
claims_export_df["service_time"] = claims_export_df["service_time"].astype(str)

facilities_df.to_parquet("facilities.parquet", index=False)
doctors_df.to_parquet("doctors.parquet", index=False)
patients_df.to_parquet("patients.parquet", index=False)
claims_export_df.to_parquet("claims.parquet", index=False)

print("Exported: facilities.parquet, doctors.parquet, patients.parquet, claims.parquet")

Exported: facilities.parquet, doctors.parquet, patients.parquet, claims.parquet


## 6. Construct Heterogeneous Graph (`HeteroData`)

We model the healthcare graph with 5 entity node types and relational edges:
- `('patient', 'registered_at', 'facility')`
- `('patient', 'examined_by', 'doctor')`
- `('doctor', 'authorized', 'claim')`
- `('claim', 'has_diagnosis', 'diagnosis')`
- `('patient', 'referred_to', 'facility')`

In [29]:
import torch
import numpy as np
import pandas as pd
from torch_geometric.data import HeteroData
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import torch_geometric.transforms as T
from sklearn.model_selection import train_test_split

print("Building structured HeteroData graph with advanced ML methodology...")
hetero_data = HeteroData()

# 1. Map IDs to continuous integer indices
patient_ids_unique = patients_df["patient_id"].unique()
doctor_ids_unique = doctors_df["doctor_id"].unique()
facility_ids_unique = facilities_df["facility_id"].unique()
claim_ids_unique = claims_df["claim_id"].unique()
diagnosis_ids_unique = claims_df["icd10_code"].unique()

patient_to_idx = {pid: i for i, pid in enumerate(patient_ids_unique)}
doctor_to_idx = {did: i for i, did in enumerate(doctor_ids_unique)}
facility_to_idx = {fid: i for i, fid in enumerate(facility_ids_unique)}
claim_to_idx = {cid: i for i, cid in enumerate(claim_ids_unique)}
diagnosis_to_idx = {did: i for i, did in enumerate(diagnosis_ids_unique)}

# 2. Stratified train/val/test split (70% train, 15% val, 15% test) based on label 'is_fraud' to prevent data leakage
y_labels = claims_df["is_fraud"].to_numpy()
claim_indices = np.arange(len(y_labels))

train_idx, temp_idx = train_test_split(
    claim_indices, test_size=0.30, random_state=RANDOM_SEED, stratify=y_labels
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, random_state=RANDOM_SEED, stratify=y_labels[temp_idx]
)

train_mask = torch.zeros(len(y_labels), dtype=torch.bool)
val_mask = torch.zeros(len(y_labels), dtype=torch.bool)
test_mask = torch.zeros(len(y_labels), dtype=torch.bool)

train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True

# 3. Extract and Build Feature Tensors (x) for Nodes (Leakage Free)
# --- Patient Features ---
pat_region_enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
pat_region_features = pat_region_enc.fit_transform(patients_df[["home_region"]])
hetero_data["patient"].x = torch.tensor(pat_region_features, dtype=torch.float)
hetero_data["patient"].node_id = torch.arange(len(patient_ids_unique), dtype=torch.long)

# --- Doctor Features ---
doc_scalar = StandardScaler()
doc_hours_scaled = doc_scalar.fit_transform(doctors_df[["schedule_hours"]])
doc_spec_enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
doc_spec_features = doc_spec_enc.fit_transform(doctors_df[["specialty"]])
doc_features = np.hstack([doc_hours_scaled, doc_spec_features])
hetero_data["doctor"].x = torch.tensor(doc_features, dtype=torch.float)
hetero_data["doctor"].node_id = torch.arange(len(doctor_ids_unique), dtype=torch.long)

# --- Facility Features ---
fac_scaler = StandardScaler()
fac_geo_scaled = fac_scaler.fit_transform(facilities_df[["latitude", "longitude"]])
fac_class_enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
fac_class_features = fac_class_enc.fit_transform(facilities_df[["facility_class"]])
fac_features = np.hstack([fac_geo_scaled, fac_class_features])
hetero_data["facility"].x = torch.tensor(fac_features, dtype=torch.float)
hetero_data["facility"].node_id = torch.arange(len(facility_ids_unique), dtype=torch.long)

# --- Claim Features (Prevent Leakage with Split-based Scaling and Rich Features) ---
# Extracted time of service into numerical hours
claims_df_processed = claims_df.copy()
claims_df_processed["service_hour"] = claims_df_processed["service_time"].apply(lambda t: t.hour if hasattr(t, 'hour') else int(str(t).split(':')[0]))

# Fitting scalers and encoders only on train mask
clm_scaler = StandardScaler()
clm_scaler.fit(claims_df_processed.loc[train_idx, ["ina_cbgs_tariff", "length_of_stay", "service_hour"]])
clm_num_scaled = clm_scaler.transform(claims_df_processed[["ina_cbgs_tariff", "length_of_stay", "service_hour"]])

clm_icd10_enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
clm_icd10_enc.fit(claims_df_processed.loc[train_idx, ["icd10_code"]])
clm_icd10_features = clm_icd10_enc.transform(claims_df_processed[["icd10_code"]])

clm_icd9_enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
clm_icd9_enc.fit(claims_df_processed.loc[train_idx, ["icd9_code"]])
clm_icd9_features = clm_icd9_enc.transform(claims_df_processed[["icd9_code"]])

clm_features = np.hstack([clm_num_scaled, clm_icd10_features, clm_icd9_features])
hetero_data["claim"].x = torch.tensor(clm_features, dtype=torch.float)
hetero_data["claim"].node_id = torch.arange(len(claim_ids_unique), dtype=torch.long)

# --- Diagnosis Features ---
diag_features = np.eye(len(diagnosis_ids_unique))
hetero_data["diagnosis"].x = torch.tensor(diag_features, dtype=torch.float)
hetero_data["diagnosis"].node_id = torch.arange(len(diagnosis_ids_unique), dtype=torch.long)

# 4. Build Logic-Aligned Relational Edges
# --- patient -registered_at-> facility ---
regions = sorted(patients_df["home_region"].unique())
region_to_fac = {reg: facility_ids_unique[i % len(facility_ids_unique)] for i, reg in enumerate(regions)}
patient_reg_fac_idx = patients_df["home_region"].map(region_to_fac).fillna(facility_ids_unique[0])
src_pat_reg = [patient_to_idx[p] for p in patients_df["patient_id"]]
dst_pat_reg = [facility_to_idx[f] for f in patient_reg_fac_idx]
hetero_data["patient", "registered_at", "facility"].edge_index = torch.tensor(
    [src_pat_reg, dst_pat_reg], dtype=torch.long
)

# --- patient -examined_by-> doctor ---
src_pat_exam = claims_df["patient_id"].map(patient_to_idx).to_numpy()
dst_doc_exam = claims_df["doctor_id"].map(doctor_to_idx).to_numpy()
hetero_data["patient", "examined_by", "doctor"].edge_index = torch.tensor(
    [src_pat_exam, dst_doc_exam], dtype=torch.long
)

# --- doctor -authorized-> claim ---
src_doc_auth = claims_df["doctor_id"].map(doctor_to_idx).to_numpy()
dst_clm_auth = claims_df["claim_id"].map(claim_to_idx).to_numpy()
hetero_data["doctor", "authorized", "claim"].edge_index = torch.tensor(
    [src_doc_auth, dst_clm_auth], dtype=torch.long
)

# --- claim -has_diagnosis-> diagnosis ---
src_clm_diag = claims_df["claim_id"].map(claim_to_idx).to_numpy()
dst_diag = claims_df["icd10_code"].map(diagnosis_to_idx).to_numpy()
hetero_data["claim", "has_diagnosis", "diagnosis"].edge_index = torch.tensor(
    [src_clm_diag, dst_diag], dtype=torch.long
)

# --- patient -referred_to-> facility (Corrected mapping across all claim entries without drop_duplicates) ---
src_pat_ref = []
dst_pat_ref = []

for _, row in claims_df.iterrows():
    src_pat_ref.append(patient_to_idx[row["patient_id"]])
    dst_pat_ref.append(facility_to_idx[row["facility_id"]])

# Fallback/Assign registered regions for patients who have no claims
patients_with_claims = set(claims_df["patient_id"].unique())
for _, row in patients_df.iterrows():
    p_id = row["patient_id"]
    if p_id not in patients_with_claims:
        src_pat_ref.append(patient_to_idx[p_id])
        fallback_facility = region_to_fac.get(row["home_region"], facility_ids_unique[0])
        dst_pat_ref.append(facility_to_idx[fallback_facility])

hetero_data["patient", "referred_to", "facility"].edge_index = torch.tensor(
    [src_pat_ref, dst_pat_ref], dtype=torch.long
)

# Labels, attributes, and splits
hetero_data["claim"].y = torch.tensor(y_labels, dtype=torch.long)
hetero_data["claim"].fraud_scenario = claims_df["fraud_scenario"].fillna("").to_numpy()
hetero_data["claim"].train_mask = train_mask
hetero_data["claim"].val_mask = val_mask
hetero_data["claim"].test_mask = test_mask

print("Constructed HeteroData Graph successfully with robust validation setups:")
print(hetero_data)

Building structured HeteroData graph with advanced ML methodology...
Constructed HeteroData Graph successfully with robust validation setups:
HeteroData(
  patient={
    x=[5000, 34],
    node_id=[5000],
  },
  doctor={
    x=[150, 6],
    node_id=[150],
  },
  facility={
    x=[30, 4],
    node_id=[30],
  },
  claim={
    x=[19999, 23],
    node_id=[19999],
    y=[19999],
    fraud_scenario=[19999],
    train_mask=[19999],
    val_mask=[19999],
    test_mask=[19999],
  },
  diagnosis={
    x=[10, 10],
    node_id=[10],
  },
  (patient, registered_at, facility)={ edge_index=[2, 5000] },
  (patient, examined_by, doctor)={ edge_index=[2, 19999] },
  (doctor, authorized, claim)={ edge_index=[2, 19999] },
  (claim, has_diagnosis, diagnosis)={ edge_index=[2, 19999] },
  (patient, referred_to, facility)={ edge_index=[2, 20109] }
)


## 7. Serialize & Verify HeteroData Graph

In [30]:
import torch_geometric.transforms as T
import torch

print("Applying ToUndirected transformation to clean the graph for training...")
undirected_transform = T.ToUndirected()
final_hetero_data = undirected_transform(hetero_data)

# Serialize and Verify
output_path = "aura_jkn_hetero_graph.pt"
torch.save(final_hetero_data, output_path)
print(f"Successfully saved processed heterogeneous graph to '{output_path}'.")

loaded_hetero_data = torch.load(output_path, weights_only=False)
assert loaded_hetero_data["claim"].y.sum().item() == hetero_data["claim"].y.sum().item()
print("Verification successful: Reloaded graph matches in-memory transform.")
print(loaded_hetero_data)

Applying ToUndirected transformation to clean the graph for training...
Successfully saved processed heterogeneous graph to 'aura_jkn_hetero_graph.pt'.
Verification successful: Reloaded graph matches in-memory transform.
HeteroData(
  patient={
    x=[5000, 34],
    node_id=[5000],
  },
  doctor={
    x=[150, 6],
    node_id=[150],
  },
  facility={
    x=[30, 4],
    node_id=[30],
  },
  claim={
    x=[19999, 23],
    node_id=[19999],
    y=[19999],
    fraud_scenario=[19999],
    train_mask=[19999],
    val_mask=[19999],
    test_mask=[19999],
  },
  diagnosis={
    x=[10, 10],
    node_id=[10],
  },
  (patient, registered_at, facility)={ edge_index=[2, 5000] },
  (patient, examined_by, doctor)={ edge_index=[2, 19999] },
  (doctor, authorized, claim)={ edge_index=[2, 19999] },
  (claim, has_diagnosis, diagnosis)={ edge_index=[2, 19999] },
  (patient, referred_to, facility)={ edge_index=[2, 20109] },
  (facility, rev_registered_at, patient)={ edge_index=[2, 5000] },
  (doctor, rev_exa